# 01 · SEIR baseline (เงื่อนไข A) และ rule-based behaviour (เงื่อนไข B)

ตรวจกลไกโรคก่อนใส่ LLM (แนวทางระยะที่ 2 และหัวข้อ 9.3 ข้อ 1)

$$\frac{dS}{dt}=-\beta_{\mathrm{eff},t}\frac{SI}{N},\quad \frac{dE}{dt}=\beta_{\mathrm{eff},t}\frac{SI}{N}-\sigma E,\quad \frac{dI}{dt}=\sigma E-\gamma I,\quad \frac{dR}{dt}=\gamma I$$

- $\sigma = 1/\text{latent\_days}$, $\gamma = 1/\text{infectious\_days}$ เป็น **อัตราต่อวัน** (แก้ความไม่สอดคล้องระหว่างเปเปอร์ที่ใช้ "ระยะเวลา")
- ผู้ป่วยที่รายงานรายวัน = การไหลจาก E → I ในวันนั้น
- $\beta_{\mathrm{eff}} = \beta_0\,[w_{home} + (1-w_{home})\,q_{out}\,q_{policy}]$ : A ใช้ $q_{out}=1$, B ใช้ $q_{out}=1/(1+\alpha\cdot\text{cases}_7)$

ต้องรัน notebook 00 ก่อน (ต้องมี `data/processed/contact_by_location_TH.csv`) ไม่ต้องใช้ GPU

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
import copy
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gabm import seir
from gabm.sim import run_simulation, contact_model_from_cfg
from gabm.analysis import load_daily, load_runs, epi_metrics, plot_epidemic

N = cfg["population"]["N"]
beta0, sigma, gamma = seir.rates(cfg["disease"])
R0 = beta0 / gamma
print(f"beta0={beta0}/day  sigma={sigma:.3f}/day  gamma={gamma:.3f}/day  R0={R0:.2f}  expected final size={seir.final_size(R0):.3f}")

## 1. ตรวจระบบ: conservation, β₀ = 0 และ final size

In [ ]:
long = pd.DataFrame(seir.run(cfg, days=730))
total = long[["S", "E", "I", "R"]].sum(axis=1)
print(f"max |S+E+I+R-N| = {np.abs(total - N).max():.2e}")

# Final size: start mostly susceptible; with a small seed the attack rate approaches the final-size equation
z_sim = (N - long["S"].iloc[-1]) / N
z_theory = seir.final_size(R0)
print(f"final attack rate: simulated {z_sim:.3f} vs theory {z_theory:.3f}")
assert abs(z_sim - z_theory) < 0.01, "final size check failed"

no_tx = pd.DataFrame(seir.run(cfg, days=60, beta_fn=lambda d, s: 0.0))
assert no_tx["new_infections"].max() < 1e-9 and no_tx["I"].iloc[-1] < no_tx["I"].iloc[0], "beta0=0 should give no new infections"
print("beta0 = 0: no new infections, I decays  ✓")

fig, ax = plt.subplots(figsize=(9, 3.5))
for k, col in zip("SEIR", ["#4C78A8", "#F58518", "#E45756", "#54A24B"]):
    ax.plot(long["day"], long[k] / N, label=k, color=col)
ax.set_xlabel("day"); ax.set_ylabel("fraction of N"); ax.legend(frameon=False); ax.set_title(f"Condition A, 2 years, R0={R0:.1f}")
plt.tight_layout(); plt.savefig(FIG / "01_seir_long.png", dpi=150); plt.show()

## 2. Sweep β₀ เทียบกับสมการ final size

In [ ]:
rows = []
for b in [0.12, 0.15, 0.2, 0.25, 0.3]:
    c2 = copy.deepcopy(cfg); c2["disease"]["beta0"] = b
    out = pd.DataFrame(seir.run(c2, days=1500))
    rows.append({"beta0": b, "R0": b / gamma, "attack_sim": (N - out["S"].iloc[-1]) / N,
                 "attack_theory": seir.final_size(b / gamma), "peak_day": int(out.loc[out["I"].idxmax(), "day"])})
sweep = pd.DataFrame(rows); sweep.round(3)

## 3. เงื่อนไข A และ B ในช่วงเวลาเดียวกับ prototype

ใช้ `run_simulation` ตัวเดียวกับที่ notebook 03–05 ใช้ ผลถูกเก็บใน `outputs/runs/` เพื่อเปรียบเทียบกับ LLM residents ภายหลัง
นอกจากรอบ 30 วัน (ตาม `sim.days`) ยังรันรอบ 180 วัน (tag `long180`) เพื่อดูรูปร่างการระบาดทั้งหมด

In [ ]:
cm = contact_model_from_cfg(cfg, ROOT)
stores = [run_simulation(cfg, cond, seed=0, cm=cm, root=ROOT) for cond in "AB"]
cfg180 = copy.deepcopy(cfg); cfg180["sim"]["days"] = 180
stores += [run_simulation(cfg180, cond, seed=0, cm=cm, root=ROOT, tag="long180") for cond in "AB"]

daily = load_daily([s.run_id for s in stores], ROOT)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
d180 = daily[daily.tag == "long180"]
plot_epidemic(d180, ax=axes[0], y="I"); axes[0].set_title("Infectious I (180 days)")
plot_epidemic(d180, ax=axes[1], y="reported_per100k"); axes[1].set_title("Reported /100k/day")
plot_epidemic(d180, ax=axes[2], y="beta_eff"); axes[2].set_title("beta_eff")
plt.tight_layout(); plt.savefig(FIG / "01_A_vs_B.png", dpi=150); plt.show()

pd.DataFrame([{"condition": c, "tag": t, **epi_metrics(g, N)} for (c, t), g in daily.groupby(["condition", "tag"])]).round(4)

**คำถามตรวจความเข้าใจ**
1. ทำไม B มี peak ต่ำและช้ากว่า A ทั้งที่ $\beta_0$ เท่ากัน?
2. ถ้าเพิ่ม `w_home` (สัดส่วนการสัมผัสในบ้าน) ผลของพฤติกรรมนอกบ้านต่อ $\beta_{\mathrm{eff}}$ จะมากขึ้นหรือน้อยลง?